# PETaDex ORF Biochemical Properties Precompute

> **Author:** Angela Jiang  
> **Date Started:** 2026-07-01  
> **Date Finished:**2026-07-31  
> **Input:** s3://petadex/logan/petadex.catalytic_orfs.v1.1.fa #To be
> changed after database update **Output:** `orf_biochemical_properties`
> (PostgreSQL table)

------------------------------------------------------------------------

## 1. Objective

The goal of this pipeline is to precompute sequence-derived biochemical
properties for every PETaDex catalytic ORF and load them into a
queryable PostgreSQL table, so downstream analyses can filter and rank
candidates directly rather than recomputing properties from amino acid
strings each time.

### 1.1. Place in the Project

Properties like hydrophobicity, isoelectric point, and stability are
useful first-pass filters when triaging candidate plastic-degrading
enzymes for wet-lab validation — for example, prioritizing sequences
that are predicted stable and soluble before committing time to
expression and assay. Precomputing these once for the whole corpus means
every future analysis or notebook can join against this table instead of
re-running Biopython over hundreds of millions of sequences.

## 2. Experiment

### 2.0. Install requirements

1.  Biopython (`Bio.SeqUtils.ProtParam`)
2.  Python 3.9+ with `boto3`
3.  PostgreSQL client (`psql`) for loading

### 2.1. Tool selection

Biochemical features are computed with
`Bio.SeqUtils.ProtParam.ProteinAnalysis`, selected because it computes
deterministic sequence-derived properties without alignment, structure
prediction, or trained inference — no GPU and no external database
lookups are required.

| Output column       | Biopython call         | Rationale                                                                                          |
|------------------------|------------------------|------------------------|
| `sequence_length`   | `len(sequence)`        | Needed for length filters and mature-length calculation when joined to SignalP cleavage positions. |
| `molecular_weight`  | `.molecular_weight()`  | Full-sequence molecular weight in Daltons.                                                         |
| `isoelectric_point` | `.isoelectric_point()` | Predicted pI; relevant to solubility and purification.                                             |
| `gravy_score`       | `.gravy()`             | Grand average of hydropathicity; relevant to hydrophobicity/solubility.                            |
| `instability_index` | `.instability_index()` | Coarse sequence-derived stability estimate.                                                        |
| `aromaticity`       | `.aromaticity()`       | Fraction of aromatic residues; potentially relevant to substrate binding.                          |

**Amino acid alphabet.** ProtParam calculations require the 20 standard
residues (`ACDEFGHIKLMNPQRSTVWY`). Terminal stop symbols (`*`) are
stripped before validation. Any other character marks the row
`invalid_sequence` rather than forcing a calculation on ambiguous
residues (e.g. `X`).

### 2.2. Stream the ORF FASTA and compute properties

Every PETaDex ORF record uses the following pipe-delimited header:

``` text
>{orf_id}|{genbank_accession}|{library_id}|{contig_id}|{orf_start}|{orf_end}|{orf_type}
```

Only the first field is used:

``` python
orf_id = int(header_first_token.split("|", 1)[0])
```

All other header fields are ignored; properties are computed from
sequence alone and keyed by `orf_id`.

The pipeline (`precompute_biochem_resumable.py`, Appendix A) streams the
FASTA directly from S3 rather than downloading it first:

``` text
S3 PETaDex ORF FASTA
  -> streamed record-by-record via boto3
  -> per-sequence Biopython ProteinAnalysis calculation
  -> gzip-compressed CSV part files
  -> manifest/checkpoint JSON
  -> later PostgreSQL COPY load
```

No full FASTA or output table is held in memory; one record and one
output part are processed at a time.

**Sequence cleaning and `calc_status`.** For each record: uppercase the
sequence, strip `*`, and compare remaining characters to the standard
alphabet. If noncanonical residues remain, the row is written with
`calc_status = invalid_sequence`, a sequence length, and null feature
values. Otherwise all ProtParam properties are computed and the row is
written with `calc_status = ok`. This preserves one row per ORF and
distinguishes “not present” from “present but not computable.”

**Resumability.** Output parts are written atomically: each part is
written to `*.csv.gz.tmp` and renamed to `*.csv.gz` only once complete.
A `manifest.json` checkpoint records rows written and status counts
after each completed part. On `--resume`, the script skips records
already accounted for in the manifest and continues from the next
unfinished region — at most one in-progress part needs to be recomputed
after an interruption. Output part size is set by `--rows-per-file`
(production: `5,000,000`), balancing checkpoint frequency against file
count.

Run command:

``` bash
nohup python precompute_biochem_resumable.py \
  --fasta s3://petadex/logan/petadex.catalytic_orfs.v1.1.fa \
  --out-dir ./biochem_FULL \
  --rows-per-file 5000000 \
  --progress-every 100000 \
  --resume \
  > biochem_FULL.nohup.log 2>&1 &
```

Resuming after interruption uses the same command with the same
`--out-dir`; the manifest determines where to continue.

### 2.3. Output format

``` text
biochem_FULL/
├── manifest.json
├── bad_records.tsv
└── parts/
    ├── orf_biochem_part_000000.csv.gz
    └── ...
```

CSV columns:

``` text
orf_id,sequence_length,molecular_weight,isoelectric_point,gravy_score,instability_index,aromaticity,calc_status,noncanonical_residues
```

Null token: `\N` (matches PostgreSQL `COPY` convention).
`noncanonical_residues` is populated only for `invalid_sequence` rows.

Records where `orf_id` cannot be parsed at all (distinct from a valid
ORF with an invalid sequence) are logged to `bad_records.tsv` (`reason`,
`header`, `details`) rather than written to a CSV part.

### 2.4. Create the PostgreSQL schema

``` sql
CREATE TABLE public.orf_biochemical_properties (
    orf_id BIGINT PRIMARY KEY,
    sequence_length INTEGER NOT NULL,
    molecular_weight REAL,
    isoelectric_point REAL,
    gravy_score REAL,
    instability_index REAL,
    aromaticity REAL,
    calc_status TEXT NOT NULL DEFAULT 'ok',
    noncanonical_residues TEXT,

    CONSTRAINT orf_biochemical_properties_sequence_length_check
        CHECK (sequence_length > 0),

    CONSTRAINT orf_biochemical_properties_isoelectric_point_check
        CHECK (isoelectric_point IS NULL OR (isoelectric_point >= 0 AND isoelectric_point <= 14)),

    CONSTRAINT orf_biochemical_properties_aromaticity_check
        CHECK (aromaticity IS NULL OR (aromaticity >= 0 AND aromaticity <= 1)),

    CONSTRAINT orf_biochemical_properties_calc_status_check
        CHECK (calc_status IN ('ok', 'invalid_sequence')),

    CONSTRAINT orf_biochemical_properties_status_consistency_check
        CHECK (
            (calc_status = 'ok' AND molecular_weight IS NOT NULL AND isoelectric_point IS NOT NULL
             AND gravy_score IS NOT NULL AND instability_index IS NOT NULL AND aromaticity IS NOT NULL)
            OR
            (calc_status = 'invalid_sequence' AND molecular_weight IS NULL AND isoelectric_point IS NULL
             AND gravy_score IS NULL AND instability_index IS NULL AND aromaticity IS NULL)
        )
);
```

| Column                                                                                     | Type      | Rationale                                                                 |
|------------------------|------------------------|------------------------|
| `orf_id`                                                                                   | `BIGINT`  | Matches `orf_origins.orf_id`; avoids overflow as the corpus grows.        |
| `sequence_length`                                                                          | `INTEGER` | Exact residue count.                                                      |
| `molecular_weight`, `isoelectric_point`, `gravy_score`, `instability_index`, `aromaticity` | `REAL`    | Four-byte float sufficient for all sequence-derived values at this scale. |
| `calc_status`                                                                              | `TEXT`    | Distinguishes computed vs. intentionally-null rows.                       |
| `noncanonical_residues`                                                                    | `TEXT`    | Auditable record of observed non-standard symbols.                        |

Indexes are built after bulk load, selected by actual query pattern
(e.g. `gravy_score`, `sequence_length`, `calc_status`).

------------------------------------------------------------------------

## Appendix: Pipeline Script

### A. `precompute_biochem_resumable.py`

Streams the FASTA corpus directly from S3, computes ProtParam properties
per record, and writes resumable gzip-compressed CSV parts.

``` python
#!/usr/bin/env python3

import argparse
import csv
import gzip
import io
import json
import math
import re
import time
from pathlib import Path
from urllib.parse import urlparse

import boto3
from Bio.SeqUtils.ProtParam import ProteinAnalysis

STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")
HEADER_RE = re.compile(r"^>(\S+)")

COLUMNS = [
    "orf_id",
    "sequence_length",
    "molecular_weight",
    "isoelectric_point",
    "gravy_score",
    "instability_index",
    "aromaticity",
    "calc_status",
    "noncanonical_residues",
]


def parse_orf_id(header_line: str) -> int:
    """
    PETaDex header:
    >{orf_id}|{genbank_accession}|{library_id}|{contig_id}|{orf_start}|{orf_end}|{orf_type}
    """
    m = HEADER_RE.match(header_line.strip())
    if not m:
        raise ValueError(f"Bad FASTA header: {header_line[:120]}")
    first_token = m.group(1)
    orf_id_text = first_token.split("|", 1)[0]
    return int(orf_id_text)


def clean_seq(seq: str):
    seq = seq.strip().upper().replace("*", "")
    invalid = sorted(set(seq) - STANDARD_AA)
    return seq, invalid


def compute_props(seq: str):
    p = ProteinAnalysis(seq)
    return {
        "sequence_length": len(seq),
        "molecular_weight": p.molecular_weight(),
        "isoelectric_point": p.isoelectric_point(),
        "gravy_score": p.gravy(),
        "instability_index": p.instability_index(),
        "aromaticity": p.aromaticity(),
    }


def fmt_float(x):
    if x is None:
        return r"\N"
    if isinstance(x, float) and (math.isnan(x) or math.isinf(x)):
        return r"\N"
    return f"{x:.6f}"


def open_fasta_stream(source: str):
    """
    Open a FASTA source as a text-mode line iterator.
    Supports local paths and s3:// URIs. S3 objects are streamed directly
    (not downloaded to disk) via a boto3 StreamingBody wrapped in
    TextIOWrapper. Requires credentials available to boto3 (an EC2 instance
    profile is recommended for long-running streams, since instance
    credentials auto-refresh).
    """
    if source.startswith("s3://"):
        parsed = urlparse(source)
        bucket = parsed.netloc
        key = parsed.path.lstrip("/")
        s3 = boto3.client("s3")
        obj = s3.get_object(Bucket=bucket, Key=key)
        return io.TextIOWrapper(obj["Body"], encoding="utf-8", errors="replace")
    return open(source, "rt", encoding="utf-8", errors="replace")


def fasta_records(source: str):
    handle = open_fasta_stream(source)
    try:
        header = None
        chunks = []
        for line in handle:
            if line.startswith(">"):
                if header is not None:
                    yield header, "".join(chunks)
                header = line.rstrip("\n")
                chunks = []
            else:
                chunks.append(line.strip())
        if header is not None:
            yield header, "".join(chunks)
    finally:
        handle.close()


def load_manifest(manifest_path: Path):
    if manifest_path.exists():
        with manifest_path.open("rt") as f:
            return json.load(f)
    return None


def save_manifest(manifest_path: Path, manifest: dict):
    tmp_path = manifest_path.with_suffix(".json.tmp")
    with tmp_path.open("wt") as f:
        json.dump(manifest, f, indent=2)
    tmp_path.replace(manifest_path)


def write_part(out_dir: Path, part_index: int, rows):
    parts_dir = out_dir / "parts"
    parts_dir.mkdir(parents=True, exist_ok=True)

    final_path = parts_dir / f"orf_biochem_part_{part_index:06d}.csv.gz"
    tmp_path = parts_dir / f"orf_biochem_part_{part_index:06d}.csv.gz.tmp"

    if final_path.exists():
        return final_path

    with gzip.open(tmp_path, "wt", newline="") as fh:
        writer = csv.writer(fh)
        writer.writerow(COLUMNS)
        writer.writerows(rows)

    tmp_path.replace(final_path)
    return final_path


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--fasta", required=True, help="Local path or s3:// URI")
    ap.add_argument("--out-dir", required=True)
    ap.add_argument("--rows-per-file", type=int, default=5_000_000)
    ap.add_argument("--progress-every", type=int, default=100_000)
    ap.add_argument("--max-records", type=int, default=None)
    ap.add_argument("--resume", action="store_true")
    args = ap.parse_args()

    fasta_source = args.fasta
    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    parts_dir = out_dir / "parts"
    parts_dir.mkdir(exist_ok=True)

    manifest_path = out_dir / "manifest.json"
    bad_path = out_dir / "bad_records.tsv"

    # Clean abandoned temp files from interrupted runs.
    for tmp in parts_dir.glob("*.tmp"):
        tmp.unlink()

    old_manifest = load_manifest(manifest_path) if args.resume else None
    completed_rows = int(old_manifest.get("rows_written", 0)) if old_manifest else 0
    completed_parts = int(old_manifest.get("files_written", 0)) if old_manifest else 0

    stats = {
        "source_fasta": fasta_source,
        "rows_per_file": args.rows_per_file,
        "records_seen_total_this_run": 0,
        "records_skipped_due_to_resume": completed_rows,
        "rows_written": completed_rows,
        "ok_rows": int(old_manifest.get("ok_rows", 0)) if old_manifest else 0,
        "invalid_sequence_rows": int(old_manifest.get("invalid_sequence_rows", 0)) if old_manifest else 0,
        "parse_error_rows": int(old_manifest.get("parse_error_rows", 0)) if old_manifest else 0,
        "files_written": completed_parts,
        "start_time": time.time(),
        "resume": bool(old_manifest),
    }

    print(f"[start] fasta={fasta_source}", flush=True)
    print(f"[start] out_dir={out_dir}", flush=True)
    print(f"[start] resume={bool(old_manifest)} completed_rows={completed_rows:,}", flush=True)

    if not bad_path.exists() or not old_manifest:
        with bad_path.open("wt") as bad:
            bad.write("reason\theader\tdetails\n")

    current_rows = []
    part_index = completed_parts
    processed_index = 0

    with bad_path.open("at") as bad:
        for header, seq_raw in fasta_records(fasta_source):
            if args.max_records is not None and processed_index >= args.max_records:
                break

            processed_index += 1
            stats["records_seen_total_this_run"] += 1

            if processed_index <= completed_rows:
                if processed_index % args.progress_every == 0:
                    print(f"[resume-skip] skipped={processed_index:,}/{completed_rows:,}", flush=True)
                continue

            try:
                orf_id = parse_orf_id(header)
                seq, invalid = clean_seq(seq_raw)

                if len(seq) == 0:
                    raise ValueError("empty_sequence")

                if invalid:
                    row = [
                        orf_id, len(seq), r"\N", r"\N", r"\N", r"\N", r"\N",
                        "invalid_sequence", "".join(invalid),
                    ]
                    stats["invalid_sequence_rows"] += 1
                else:
                    props = compute_props(seq)
                    row = [
                        orf_id,
                        props["sequence_length"],
                        fmt_float(props["molecular_weight"]),
                        fmt_float(props["isoelectric_point"]),
                        fmt_float(props["gravy_score"]),
                        fmt_float(props["instability_index"]),
                        fmt_float(props["aromaticity"]),
                        "ok", "",
                    ]
                    stats["ok_rows"] += 1

                current_rows.append(row)
                stats["rows_written"] += 1

            except Exception as e:
                stats["parse_error_rows"] += 1
                safe_header = header.replace("\t", " ")[:300]
                bad.write(f"parse_error\t{safe_header}\t{repr(e)}\n")

            if len(current_rows) >= args.rows_per_file:
                final_path = write_part(out_dir, part_index, current_rows)
                part_index += 1
                current_rows = []

                stats["files_written"] = part_index
                stats["elapsed_sec"] = time.time() - stats["start_time"]
                stats["records_per_sec_this_run"] = (
                    stats["records_seen_total_this_run"] / stats["elapsed_sec"]
                    if stats["elapsed_sec"] else None
                )
                save_manifest(manifest_path, stats)

                print(
                    f"[part-complete] {final_path} "
                    f"rows_written={stats['rows_written']:,} "
                    f"ok={stats['ok_rows']:,} "
                    f"invalid={stats['invalid_sequence_rows']:,} "
                    f"errors={stats['parse_error_rows']:,}",
                    flush=True,
                )

            if processed_index % args.progress_every == 0:
                elapsed = time.time() - stats["start_time"]
                rate = stats["records_seen_total_this_run"] / elapsed if elapsed else 0
                print(
                    f"[progress] records_seen_this_run={stats['records_seen_total_this_run']:,} "
                    f"global_position={processed_index:,} "
                    f"rows_written={stats['rows_written']:,} "
                    f"rate_this_run={rate:,.1f}/sec",
                    flush=True,
                )

    if current_rows:
        final_path = write_part(out_dir, part_index, current_rows)
        part_index += 1
        stats["files_written"] = part_index
        print(f"[final-part-complete] {final_path}", flush=True)

    stats["elapsed_sec"] = time.time() - stats["start_time"]
    stats["records_per_sec_this_run"] = (
        stats["records_seen_total_this_run"] / stats["elapsed_sec"]
        if stats["elapsed_sec"] else None
    )
    stats["completed"] = True
    save_manifest(manifest_path, stats)

    print("[complete]", flush=True)
    print(json.dumps(stats, indent=2), flush=True)


if __name__ == "__main__":
    main()
```